In [1]:
import pandas as pd

In [8]:
df=pd.read_excel('../data/Gen_AI Dataset.xlsx')

In [9]:
import pandas as pd

pd.set_option("display.max_colwidth", None)


In [10]:
df.head()

,Query,Assessment_url
0,I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.,https://www.shl.com/solutions/products/product-catalog/view/automata-fix-new/
1,I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.,https://www.shl.com/solutions/products/product-catalog/view/core-java-entry-level-new/
2,I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.,https://www.shl.com/solutions/products/product-catalog/view/java-8-new/
3,I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.,https://www.shl.com/solutions/products/product-catalog/view/core-java-advanced-level-new/
4,I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.,https://www.shl.com/products/product-catalog/view/interpersonal-communications/


In [11]:
before = len(df)
df.drop_duplicates(subset=["Assessment_url"], inplace=True)
df.reset_index(drop=True, inplace=True)
after = len(df)

print(f"Removed {before - after} duplicate rows")


Removed 11 duplicate rows


In [13]:
import requests
import re
import json
import time
import pandas as pd
from bs4 import BeautifulSoup
import os
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

DATASET_PATH = "../data/Gen_AI Dataset.xlsx"

output_path = os.path.join("..", "data", "product_catalog.json")


def scrape_shl_product(url):
    response = requests.get(url, headers=HEADERS, timeout=20)
    soup = BeautifulSoup(response.text, "html.parser")

    container = soup.find("div", class_="product-catalogue")
    if container is None:
        return None

    name = container.find("h1").get_text(strip=True)

    fields = {}
    for row in container.find_all("div", class_="product-catalogue-training-calendar__row"):
        label = row.find("h4")
        value = row.find("p")
        if label and value:
            fields[label.get_text(strip=True)] = value.get_text(" ", strip=True)

    description = fields.get("Description")
    job_levels = fields.get("Job levels")
    languages = fields.get("Languages")

    duration = None
    length_text = fields.get("Assessment length")
    if length_text:
        match = re.search(r"\d+", length_text)
        if match:
            duration = int(match.group())

    text = (description or "").lower()
    if "personality" in text:
        test_type = ["Personality & Behavior"]
    elif "java" in text or "coding" in text:
        test_type = ["Knowledge & Skills"]
    else:
        test_type = ["General"]

    remote_support = container.select_one("span.catalogue__circle.-yes") is not None

    return {
        "name": name,
        "url": url,
        "description": description,
        "duration_minutes": duration,
        "job_levels": job_levels,
        "languages": languages,
        "test_type": test_type,
        "remote_support": remote_support,
        "adaptive_support": False
    }


def main():
    df = pd.read_excel(DATASET_PATH)
    urls = df["Assessment_url"].dropna().unique().tolist()

    print(f"Found {len(urls)} unique assessment URLs")

    catalog = []
    success, failed = 0, 0

    for i, url in enumerate(urls, 1):
        print(f"{i}/{len(urls)} -> {url}")

        try:
            item = scrape_shl_product(url)
            if item:
                catalog.append(item)
                success += 1
                print(f"  saved: {item['name']}")
            else:
                failed += 1
                print("  no data found")

        except Exception as e:
            failed += 1
            print(f"  error: {e}")

        time.sleep(1)





    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(catalog, f, indent=2, ensure_ascii=False)


        print("Scraping finished")
        print(f"Saved: {success}, Failed: {failed}")
       


if __name__ == "__main__":
    main()


Found 54 unique assessment URLs
1/54 -> https://www.shl.com/solutions/products/product-catalog/view/automata-fix-new/
  saved: Automata - Fix (New)
2/54 -> https://www.shl.com/solutions/products/product-catalog/view/core-java-entry-level-new/
  saved: Core Java (Entry Level) (New)
3/54 -> https://www.shl.com/solutions/products/product-catalog/view/java-8-new/
  saved: Java 8 (New)
4/54 -> https://www.shl.com/solutions/products/product-catalog/view/core-java-advanced-level-new/
  saved: Core Java (Advanced Level) (New)
5/54 -> https://www.shl.com/products/product-catalog/view/interpersonal-communications/
  saved: Interpersonal Communications
6/54 -> https://www.shl.com/solutions/products/product-catalog/view/entry-level-sales-7-1/
  saved: Entry level Sales 7.1 (International)
7/54 -> https://www.shl.com/solutions/products/product-catalog/view/entry-level-sales-sift-out-7-1/
  saved: Entry Level Sales Sift Out 7.1
8/54 -> https://www.shl.com/solutions/products/product-catalog/view/entr

In [ ]:
import json

def view_catalog(limit=5):
    with open("../data/product_catalog.json", "r", encoding="utf-8") as f:
        catalog = json.load(f)

    print(f"Total assessments in catalog: {len(catalog)}\n")

    for i, item in enumerate(catalog[:limit], start=1):
        print(f"Assessment {i}")
        print(f"Name            : {item.get('name')}")
        print(f"URL             : {item.get('url')}")
        print(f"Duration        : {item.get('duration_minutes')} minutes")
        print(f"Test Type       : {item.get('test_type')}")
        print(f"Remote Support  : {item.get('remote_support')}")
        print(f"Adaptive Support: {item.get('adaptive_support')}")

        description = item.get("description")
        if description:
            print(f"Description     : {description[:300]}...")
        else:
            print("Description     : None")

        print("-" * 80)


In [11]:
view_catalog(limit=54)


Total assessments in catalog: 54

Assessment 1
Name            : Automata - Fix (New)
URL             : https://www.shl.com/solutions/products/product-catalog/view/automata-fix-new/
Duration        : 20 minutes
Test Type       : ['Knowledge & Skills']
Remote Support  : True
Adaptive Support: False
Description     : A simulated compiler integrated test to measure debugging skills in C, C++ and Java. The test checks the ability to fix logical or syntactical errors and to reuse an existing code. Your use of this assessment product may be subject to New York City Law 144 (Regulation of the Use of Automated Employm...
--------------------------------------------------------------------------------
Assessment 2
Name            : Core Java (Entry Level) (New)
URL             : https://www.shl.com/solutions/products/product-catalog/view/core-java-entry-level-new/
Duration        : 13 minutes
Test Type       : ['Knowledge & Skills']
Remote Support  : True
Adaptive Support: False
Description    

In [13]:
# !pip install sentence-transformers faiss-cpu


In [14]:
import json

with open("../data/product_catalog.json", "r", encoding="utf-8") as f:
    catalog = json.load(f)

print(f"Loaded {len(catalog)} assessments")


Loaded 54 assessments


In [15]:
def build_assessment_text(item):
    parts = [
        item.get("name", ""),
        item.get("description", ""),
        f"Job Levels: {item.get('job_levels', '')}",
        f"Languages: {item.get('languages', '')}",
        f"Test Type: {', '.join(item.get('test_type', []))}"
    ]
    return " ".join(parts)


In [16]:
documents = [build_assessment_text(item) for item in catalog]


In [17]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(doc_embeddings.shape)


2025-12-15 19:57:06.795205: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-15 19:57:06.858908: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765828626.887286    7374 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765828626.896352    7374 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765828626.952531    7374 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(54, 384)


In [18]:
import faiss

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print(f"FAISS index size: {index.ntotal}")


FAISS index size: 54


In [19]:
def recommend_assessments(query, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for idx in indices[0]:
        results.append(catalog[idx])

    return results


In [20]:
query = "I am hiring Java developers who can collaborate with business teams. The test should be around 40 minutes."

results = recommend_assessments(query, top_k=5)

for i, r in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Name:", r["name"])
    print("Duration:", r["duration_minutes"])
    print("Test Type:", r["test_type"])
    print("URL:", r["url"])



Result 1
Name: Java 8 (New)
Duration: 18
Test Type: ['Knowledge & Skills']
URL: https://www.shl.com/solutions/products/product-catalog/view/java-8-new/

Result 2
Name: Core Java (Advanced Level) (New)
Duration: 13
Test Type: ['Knowledge & Skills']
URL: https://www.shl.com/solutions/products/product-catalog/view/core-java-advanced-level-new/

Result 3
Name: Core Java (Entry Level) (New)
Duration: 13
Test Type: ['Knowledge & Skills']
URL: https://www.shl.com/solutions/products/product-catalog/view/core-java-entry-level-new/

Result 4
Name: JavaScript (New)
Duration: 9
Test Type: ['Knowledge & Skills']
URL: https://www.shl.com/solutions/products/product-catalog/view/javascript-new/

Result 5
Name: Manual Testing (New)
Duration: 10
Test Type: ['General']
URL: https://www.shl.com/solutions/products/product-catalog/view/manual-testing-new/


In [21]:

import json
import re
import numpy as np
import pandas as pd
from collections import defaultdict
ground_truth = defaultdict(set)

def recommend_urls(query, top_k=5):
    results = recommend_assessments(query, top_k)
    return [r["url"] for r in results]


for _, row in df.iterrows():
    ground_truth[row["Query"]].add(row["Assessment_url"])

print("Unique queries:", len(ground_truth))


def recall_at_k(ground_truth, k=5):
    hits = 0
    total = len(ground_truth)

    for query, true_urls in ground_truth.items():
        predicted_urls = recommend_urls(query, top_k=k)

        if any(url in predicted_urls for url in true_urls):
            hits += 1

    return hits / total


for k in [1, 3, 5, 10]:
    score = recall_at_k(ground_truth, k)
    print(f"Recall@{k}: {score:.4f}")



def show_failures(k=5, limit=5):
    shown = 0

    for query, true_urls in ground_truth.items():
        predicted_urls = recommend_urls(query, top_k=k)

        if not any(url in predicted_urls for url in true_urls):
            print("\nQuery:", query)
            print("Expected URLs:", list(true_urls))
            print("Predicted URLs:", predicted_urls)
            shown += 1

        if shown >= limit:
            break

Unique queries: 10
Recall@1: 0.6000
Recall@3: 0.8000
Recall@5: 0.9000
Recall@10: 1.0000


In [23]:
import os
import numpy as np
import faiss


EMBEDDING_DIR = "../data/embeddings"
EMBEDDINGS_PATH = "../data/embeddings/catalog_embeddings.npy"
FAISS_INDEX_PATH = "../data/embeddings/faiss.index"

os.makedirs(EMBEDDING_DIR, exist_ok=True)


np.save(EMBEDDINGS_PATH, doc_embeddings)

faiss.write_index(index, FAISS_INDEX_PATH)

print("Saved embeddings and FAISS index")


Saved embeddings and FAISS index
